# Text–audio alignment with frozen encoders

Train projection heads on MusicCaps audio–caption pairs and evaluate them on Song Describer. This is a curated version of Manuel Castillo's Lab 10 for Machine Learning for Music at Universitat Pompeu Fabra.

The audio and text encoders are frozen. Linear, MLP and transformer heads are compared. The transformer operates on slices of a pooled embedding, not on audio frames.

Install the dependencies first. Running the data and encoder cells downloads datasets and model weights. The repository contains neither. Historical results are in [results.md](../docs/results.md); this edited notebook has not been retrained during curation.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "music_experiments").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the repository.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from music_experiments.cache import cache_digest, source_row_indices


## Configuration

In [ ]:
from pathlib import Path
import os, sys, json, time, math, random, platform
from datetime import datetime

CFG = {
    # reproducibility
    "seed": 7,
    "musiccaps_revision": None,
    "songdescriber_revision": None,

    # datasets
    "musiccaps_name": "CLAPv2/MusicCaps",
    "songdescriber_name": "renumics/song-describer-dataset",

    # encoders (fixed), embeddings cached
    "audio_repr_model_id": "mtg-upf/clap_omarrq_mp_small_music",
    "audio_repr_layers": [6],
    "target_sr": 24000,
    "clip_s": 2.0,
    "text_model_id": "sentence-transformers/all-mpnet-base-v2",

    # training
    "batch_size": 128,
    "epochs": 120,
    "lr": 2e-4,
    "weight_decay": 1e-3,
    "patience": 6,
    "eval_every": 1,
    "amp": True,

    # model head, options: linear, mlp, tx
    "head_type": "mlp",
    "proj_dim": 256,
    "mlp_hidden": 512,
    "mlp_layers": 2,
    "dropout": 0.15,

    # transformer head (head_type="tx")
    "tx_tokens": 16,
    "tx_layers": 2,
    "tx_heads": 4,
    "tx_ff": 512,

    # contrastive
    "learnable_temperature": True,
    "init_temperature": 0.07,
    "label_smoothing": 0.0,

    # caching
    "cache_embeddings": True,
    "cache_fp16": True,

    # resource + emissions
    "resource_poll_s": 1.0,
    "track_emissions": False,
    "codecarbon_country_iso_code": os.getenv("CODECARBON_COUNTRY_ISO_CODE", "ESP"),

    # stability
    "grad_clip_norm": 1.0,
}

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")


## Device, seed and output directories

In [ ]:
import numpy as np
import torch

# reproducibility
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

WORK_DIR = REPO_ROOT / "artifacts" / "text_audio"
WORK_DIR.mkdir(parents=True, exist_ok=True)

# shared cache directory
CACHE_DIR = WORK_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# run specific artifacts directory
RUN_DIR = WORK_DIR / "runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

# device selection: cuda > mps > cpu
def pick_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = pick_device()
set_seed(CFG["seed"])

print("WORK_DIR :", WORK_DIR)
print("RUN_ID   :", RUN_ID)
print("RUN_DIR  :", RUN_DIR)
print("DEVICE   :", device)
print("torch    :", torch.__version__)


## Imports

In [ ]:
from sentence_transformers import SentenceTransformer
from codecarbon import OfflineEmissionsTracker
import datasets, psutil, torchaudio, omar_rq
import soundfile as sf


## Resource tracking

Measure embedding extraction and the main training run separately. RAM refers to process RSS; CPU is system-wide. CodeCarbon is optional and disabled by default. The head comparison is not included in the main run's timing. These measurements are not comparable with Lab 9's system RAM samples.

In [ ]:
import psutil
from dataclasses import dataclass, field

# resource + accelerator memory logging
@dataclass
class ResourceLog:
    ts: list = field(default_factory=list)
    cpu_pct: list = field(default_factory=list)
    ram_mb: list = field(default_factory=list)
    accel_mem_mb: list = field(default_factory=list)

def _accel_mem_mb():
    if torch.cuda.is_available():
        return float(torch.cuda.max_memory_allocated() / (1024**2))
    if device.type == "mps" and hasattr(torch, "mps") and hasattr(torch.mps, "current_allocated_memory"):
        try:
            return float(torch.mps.current_allocated_memory() / (1024**2))
        except Exception:
            return 0.0
    return 0.0

def start_resource_monitor(poll_s=1.0):
    import threading
    log = ResourceLog()
    t0 = time.time()
    stop_flag = {"stop": False}

    def run():
        p = psutil.Process(os.getpid())
        while not stop_flag["stop"]:
            now = time.time() - t0
            cpu = psutil.cpu_percent(interval=None)
            ram = p.memory_info().rss / (1024**2)
            amem = _accel_mem_mb()
            log.ts.append(now)
            log.cpu_pct.append(cpu)
            log.ram_mb.append(ram)
            log.accel_mem_mb.append(amem)
            time.sleep(poll_s)

    th = threading.Thread(target=run, daemon=True)
    th.start()
    return log, stop_flag

def stop_resource_monitor(stop_flag):
    stop_flag["stop"] = True

emissions_tracker = None
resource_log, resource_stop = None, None

# unified start/stop tracking (resources + emissions)
def start_tracking():
    global emissions_tracker, resource_log, resource_stop
    resource_log, resource_stop = start_resource_monitor(CFG["resource_poll_s"])

    if CFG["track_emissions"]:
        from codecarbon import OfflineEmissionsTracker
        emissions_tracker = OfflineEmissionsTracker(
            country_iso_code=CFG["codecarbon_country_iso_code"],
            output_dir=str(WORK_DIR),
            output_file="emissions.csv",
            log_level="error",
        )
        emissions_tracker.start()

    return time.time()

def stop_tracking(t0):
    global emissions_tracker, resource_log, resource_stop

    wall = time.time() - t0
    emissions = None

    if emissions_tracker is not None:
        emissions = emissions_tracker.stop()

    stop_resource_monitor(resource_stop)

    summary = {
        "wall_time_s": float(wall),
        "emissions_kg": None if emissions is None else float(emissions),
        "cpu_mean_pct": float(np.mean(resource_log.cpu_pct)) if resource_log.cpu_pct else None,
        "ram_max_mb": float(np.max(resource_log.ram_mb)) if resource_log.ram_mb else None,
        "accel_mem_max_mb": float(np.max(resource_log.accel_mem_mb)) if resource_log.accel_mem_mb else None,
    }

    (RUN_DIR / "resource_emissions_summary.json").write_text(
        json.dumps(summary, indent=2),
        encoding="utf-8"
    )

    return summary

print("Tracking ready.")


## Datasets

Load `CLAPv2/MusicCaps` and `renumics/song-describer-dataset` from Hugging Face. The saved run used 4,816 MusicCaps training rows, 536 validation rows and 746 Song Describer rows. Revision IDs were not saved in the original run. Set the revision fields above to pin future runs.

Song Describer has multiple captions for some tracks. The retained historical metric counts only the matching row as a positive; it is not track-level retrieval with multiple accepted captions.

In [ ]:
from datasets import load_dataset, Audio

# dataset loading
musiccaps = load_dataset(CFG["musiccaps_name"], revision=CFG["musiccaps_revision"])
songdesc = load_dataset(CFG["songdescriber_name"], revision=CFG["songdescriber_revision"])

print("MusicCaps splits:", musiccaps.keys())
print("SongDescriber splits:", songdesc.keys())


## Audio preprocessing

Convert to mono, resample to 24 kHz, then take the first two seconds (or pad short inputs). This short crop can miss the material described in a full-track caption.

In [ ]:
import torchaudio

# audio preprocessing (mono, resample, fixed length clip)
def to_mono(w):
    if w.ndim == 2 and w.shape[0] > 1:
        return w.mean(dim=0, keepdim=True)
    if w.ndim == 1:
        return w.unsqueeze(0)
    return w

def crop_or_pad(w, sr, clip_s):
    n = int(sr * clip_s)
    if w.shape[-1] >= n:
        return w[..., :n]
    pad = n - w.shape[-1]
    return torch.nn.functional.pad(w, (0, pad))

def preprocess_audio(arr, sr):
    w = torch.tensor(arr, dtype=torch.float32)
    w = to_mono(w)
    if sr != CFG["target_sr"]:
        w = torchaudio.functional.resample(w, sr, CFG["target_sr"])
    w = crop_or_pad(w, CFG["target_sr"], CFG["clip_s"])
    return w.squeeze(0)


## Frozen encoders

In [ ]:
from sentence_transformers import SentenceTransformer
import omar_rq
import torch

# load frozen text/audio encoders for embedding extraction
def get_text_encoder():
    m = SentenceTransformer(CFG["text_model_id"], device=str(device))
    m.eval()
    return m

def get_audio_repr():
    rep = omar_rq.get_model(model_id=CFG["audio_repr_model_id"], device=str(device))
    try:
        rep.eval()
    except Exception:
        pass
    return rep

text_encoder = get_text_encoder()
audio_repr = get_audio_repr()

print("Text encoder:", CFG["text_model_id"])
print("Audio repr   :", CFG["audio_repr_model_id"], "layers", CFG["audio_repr_layers"])


In [ ]:
import torch

@torch.no_grad()
def embed_text_batch(texts):
    # Encode a list of strings into a tensor with shape (B, Dt)."""
    emb = text_encoder.encode(
        texts,
        batch_size=CFG["batch_size"],
        convert_to_tensor=True,
        normalize_embeddings=False,
        show_progress_bar=False,
    )
    return emb

@torch.no_grad()
def embed_audio_batch(wavs):
    """Encode waveforms into a tensor with shape (B, Da) using omar_rq.

    Notes:
    * omar_rq expects audio shaped as (B, N), where B is batch size and N is number of samples.
    * Extra channel dimensions are collapsed to mono before embedding extraction.
    """
    wavs = wavs.to(device).float()

    # input shape normalization
    if wavs.ndim == 1:
        wavs = wavs.unsqueeze(0)
    elif wavs.ndim == 2:
        pass
    elif wavs.ndim == 3:
        if wavs.shape[1] == 1:
            wavs = wavs.squeeze(1)
        else:
            wavs = wavs.mean(dim=1)
    else:
        raise ValueError(f"Unexpected wavs shape: {tuple(wavs.shape)}")

    B = wavs.shape[0]

    # embedding extraction
    x = audio_repr.extract_embeddings(wavs, layers=CFG["audio_repr_layers"])

    # unwrap encoder output
    if isinstance(x, dict):
        k = CFG["audio_repr_layers"][0]
        x = x[k] if k in x else next(iter(x.values()))
    elif isinstance(x, (list, tuple)):
        x = x[0]

    if not torch.is_tensor(x):
        raise TypeError(f"audio_repr.extract_embeddings returned: {type(x)}")

    # batch dimension alignment
    if x.shape[0] != B and x.ndim > 1 and x.shape[1] == B:
        x = x.transpose(0, 1)

    # pooling to (B, D)
    if x.ndim == 1:
        x = x.unsqueeze(0)
    if x.ndim > 2:
        reduce_dims = tuple(range(1, x.ndim - 1))
        if reduce_dims:
            x = x.mean(dim=reduce_dims)

    if x.ndim != 2:
        raise ValueError(f"Expected (B, D) after pooling, got {tuple(x.shape)}")
    if x.shape[0] != B:
        raise ValueError(f"Batch mismatch: expected B={B}, got {tuple(x.shape)}")

    return x


## Dataset splits

Use train/validation splits if supplied. Otherwise shuffle the first MusicCaps split with seed 7 and use 90/10. Song Describer is used for evaluation. This is the notebook's experimental protocol, not a claim of an official benchmark split.

In [ ]:
# dataset splits and text field selection
if "train" in musiccaps and "validation" in musiccaps:
    mc_train = musiccaps["train"]
    mc_val = musiccaps["validation"]
else:
    full = musiccaps[list(musiccaps.keys())[0]]
    n = len(full)
    idx = np.arange(n)
    np.random.RandomState(CFG["seed"]).shuffle(idx)
    cut = int(0.9 * n)
    mc_train = full.select(idx[:cut].tolist())
    mc_val = full.select(idx[cut:].tolist())

def pick_text_col(ds):
    # pick the first matching text column name
    for c in ["caption", "text", "description"]:
        if c in ds.column_names:
            return c
    raise ValueError("No text column found.")

mc_text_col = pick_text_col(mc_train)

sd_split = "train" if "train" in songdesc else list(songdesc.keys())[0]
sd_eval = songdesc[sd_split]
sd_text_col = pick_text_col(sd_eval)

print("MusicCaps:", len(mc_train), len(mc_val), "text_col:", mc_text_col)
print("SongDesc :", len(sd_eval), "text_col:", sd_text_col, "split:", sd_split)


## Embedding extraction and cache

Cache identity includes dataset fingerprints, encoder IDs, sample rate, clip duration and layers. Decode failures are counted and recorded with source row indices. Kept row indices are used in the retrieval demo, so skipping audio does not shift the captions.

Changing model weights under the same model ID still requires clearing the cache. For a benchmark, record immutable model revisions as well as dataset revisions.

In [ ]:
import os, warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore", message="huggingface/tokenizers: The current process just got forked.*")

from tqdm.auto import tqdm
import torch
import torchaudio
import io

import soundfile as sf

# IMPORTANT: to avoid datasets trying to decode audio via torchcodec
from datasets import Audio as HF_Audio

CACHE_VER = "curated-v1"

def cache_path(name, split, ds, text_key):
    identity = cache_digest({
        "version": CACHE_VER, "dataset": name, "split": split,
        "fingerprint": ds._fingerprint, "rows": len(ds), "text_key": text_key,
        "audio_model": CFG["audio_repr_model_id"], "text_model": CFG["text_model_id"],
        "layers": CFG["audio_repr_layers"], "sr": CFG["target_sr"],
        "seconds": CFG["clip_s"], "fp16": CFG["cache_fp16"], "seed": CFG["seed"],
    })
    return CACHE_DIR / f"{name}_{split}_{identity}.pt"

def pick_audio_col(ds):
    for c in ["audio", "song", "track", "waveform", "wav", "mp3", "flac"]:
        if c in ds.column_names:
            return c
    if "path" in ds.column_names:
        return "path"
    try:
        feats = getattr(ds, "features", {})
        for c, f in feats.items():
            if f.__class__.__name__.lower() == "audio":
                return c
    except Exception:
        pass
    return None

def load_audio_from_path(path, target_sr, clip_s):
    wav, sr = torchaudio.load(path)
    wav = to_mono(wav)
    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)
    wav = crop_or_pad(wav, target_sr, clip_s)
    return wav.squeeze(0)

def load_audio_from_bytes(b, target_sr, clip_s):
    x, sr = sf.read(io.BytesIO(b), dtype="float32", always_2d=True)
    wav = torch.from_numpy(x.T)
    wav = to_mono(wav)
    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)
    wav = crop_or_pad(wav, target_sr, clip_s)
    return wav.squeeze(0)

def cache_ok(obj, expected_min_len=1):
    try:
        if not isinstance(obj, dict):
            return False
        if "audio" not in obj or "text" not in obj:
            return False
        A, T = obj["audio"], obj["text"]
        if not torch.is_tensor(A) or not torch.is_tensor(T):
            return False
        if A.ndim != 2 or T.ndim != 2:
            return False
        if A.shape[0] < expected_min_len or T.shape[0] < expected_min_len:
            return False
        if A.shape[0] != T.shape[0]:
            return False
        return bool(torch.isfinite(A).all() and torch.isfinite(T).all())
    except Exception:
        return False

def _disable_hf_audio_decode(ds, audio_col):
    """
    If ds has an Audio feature column, force decode=False locally to avoid torchcodec.
    Returns (possibly) a new dataset object.
    """
    try:
        feats = getattr(ds, "features", None)
        if feats is None:
            return ds
        f = feats.get(audio_col, None)
        if f is None:
            return ds
        if f.__class__.__name__.lower() == "audio":
            return ds.cast_column(audio_col, HF_Audio(decode=False))
        return ds
    except Exception:
        return ds

def build_embeddings(ds, text_key, split_name, cache_name):
    p = cache_path(cache_name, split_name, ds, text_key)

    if CFG["cache_embeddings"] and p.exists():
        obj = torch.load(p, map_location="cpu")
        if cache_ok(obj) and len(obj.get("meta", [])) == obj["audio"].shape[0]:
            source_row_indices(obj["meta"], len(ds))
            obj["_cache_hit"] = True
            obj["_cache_path"] = str(p)
            return obj
        print(f"[WARN] Invalid or misaligned cache, recomputing: {p.name}")

    audio_col = pick_audio_col(ds)
    if audio_col is None:
        raise KeyError(
            f"No audio column found in '{cache_name}/{split_name}'. Columns: {ds.column_names}"
        )

    # ✅ prevent HF datasets from decoding audio automatically (torchcodec)
    ds = _disable_hf_audio_decode(ds, audio_col)

    audio_embs, text_embs, meta = [], [], []
    skipped = []
    bs = CFG["batch_size"]
    n = len(ds)
    steps = (n + bs - 1) // bs
    it = tqdm(range(0, n, bs), total=steps, desc=f"Embeddings: {cache_name}/{split_name}", leave=True)

    for i in it:
        batch = ds[i:i + bs]
        texts = list(batch[text_key])
        col = batch[audio_col]

        ok_texts, ok_wavs, ok_meta = [], [], []
        for j, (txt, a) in enumerate(zip(texts, col)):
            try:
                if isinstance(a, dict) and "array" in a:
                    # if some split still provides decoded arrays
                    w = preprocess_audio(a["array"], a["sampling_rate"])
                elif isinstance(a, dict) and "bytes" in a and a["bytes"] is not None:
                    w = load_audio_from_bytes(a["bytes"], CFG["target_sr"], CFG["clip_s"])
                elif isinstance(a, dict) and "path" in a and isinstance(a["path"], str):
                    w = load_audio_from_path(a["path"], CFG["target_sr"], CFG["clip_s"])
                elif isinstance(a, str):
                    w = load_audio_from_path(a, CFG["target_sr"], CFG["clip_s"])
                else:
                    raise TypeError(f"Unsupported audio item type: {type(a)}")
                ok_wavs.append(w)
                ok_texts.append(txt)
                ok_meta.append({"i": int(i + j)})
            except Exception as exc:
                skipped.append({"i": int(i + j), "error_type": type(exc).__name__})
                continue

        if len(ok_texts) == 0:
            continue

        wavs = torch.stack(ok_wavs, dim=0)
        aemb = embed_audio_batch(wavs).detach().cpu()
        temb = embed_text_batch(ok_texts).detach().cpu()

        assert aemb.shape[0] == len(ok_texts), f"audio emb mismatch: {aemb.shape} vs {len(ok_texts)}"
        assert temb.shape[0] == len(ok_texts), f"text emb mismatch: {temb.shape} vs {len(ok_texts)}"

        if CFG["cache_fp16"]:
            aemb = aemb.half()
            temb = temb.half()

        audio_embs.append(aemb)
        text_embs.append(temb)
        meta.extend(ok_meta)

        it.set_postfix({"kept": len(ok_texts), "audio_col": audio_col})

    if not audio_embs:
        raise RuntimeError(f"No decodable audio in {cache_name}/{split_name}; inspect dataset access and codecs.")
    print(f"{cache_name}/{split_name}: kept {len(meta)} of {n}; skipped {len(skipped)}")
    out = {
        "skipped": skipped,
        "source_fingerprint": ds._fingerprint,
        "audio": torch.cat(audio_embs, dim=0),
        "text": torch.cat(text_embs, dim=0),
        "meta": meta,
        "_cache_hit": False,
        "_cache_path": str(p),
    }

    if out["audio"].shape[0] != out["text"].shape[0]:
        raise RuntimeError(f"Final A/T mismatch: {out['audio'].shape} vs {out['text'].shape}")

    if CFG["cache_embeddings"]:
        torch.save(out, p)

    return out

t0 = start_tracking()
try:
    mc_train_emb = build_embeddings(mc_train, mc_text_col, "train", "musiccaps")
    mc_val_emb = build_embeddings(mc_val, mc_text_col, "val", "musiccaps")
    sd_emb = build_embeddings(sd_eval, sd_text_col, "eval", "songdescriber")
finally:
    track_summary = stop_tracking(t0)
    (RUN_DIR / "embedding_resources.json").write_text(json.dumps(track_summary, indent=2))

print(
    "cache hits:",
    "mc_train", mc_train_emb.get("_cache_hit", None),
    "| mc_val", mc_val_emb.get("_cache_hit", None),
    "| sd", sd_emb.get("_cache_hit", None),
)
track_summary


## Check embedding values

In [ ]:
def nan_inf_report(x, name):
    # quick numeric sanity check for cached embeddings
    x = x.float()
    n_nan = torch.isnan(x).sum().item()
    n_inf = torch.isinf(x).sum().item()
    if n_nan or n_inf:
        raise ValueError(f"Non-finite embeddings: {name}")
    print(f"{name}: shape={tuple(x.shape)} | NaN={n_nan} | Inf={n_inf}")

nan_inf_report(mc_train_emb["audio"], "mc_train audio")
nan_inf_report(mc_train_emb["text"],  "mc_train text")
nan_inf_report(mc_val_emb["audio"],   "mc_val audio")
nan_inf_report(mc_val_emb["text"],    "mc_val text")
nan_inf_report(sd_emb["audio"],       "songdesc audio")
nan_inf_report(sd_emb["text"],        "songdesc text")


## Projection heads

Both branches project to 256 dimensions and normalize the output. The transformer head partitions a pooled embedding into 16 tokens; these are feature groups, not time steps.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# projection heads: map frozen audio/text embeddings into a shared space
class LinearHead(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.proj = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        return self.proj(x)

class MLPHead(nn.Module):
    def __init__(self, in_dim, out_dim, hidden, layers, dropout):
        super().__init__()
        mods = []
        d = in_dim
        for _ in range(layers - 1):
            mods += [
                nn.Linear(d, hidden),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.LayerNorm(hidden),
            ]
            d = hidden
        mods += [nn.Linear(d, out_dim)]
        self.net = nn.Sequential(*mods)

    def forward(self, x):
        return self.net(x)

class TxHead(nn.Module):
    def __init__(self, in_dim, out_dim, tokens, n_layers, n_heads, ff, dropout):
        super().__init__()
        assert in_dim % tokens == 0
        self.tokens = tokens
        self.dtok = in_dim // tokens

        enc_layer = nn.TransformerEncoderLayer(
            d_model=self.dtok,
            nhead=n_heads,
            dim_feedforward=ff,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.out = nn.Linear(self.dtok, out_dim)

    def forward(self, x):
        b = x.shape[0]
        t = x.view(b, self.tokens, self.dtok)
        z = self.enc(t).mean(dim=1)
        return self.out(z)

class CLAPAdapter(nn.Module):
    def __init__(self, audio_in, text_in, proj_dim):
        super().__init__()

        head_type = CFG["head_type"]
        if head_type == "linear":
            self.audio_head = LinearHead(audio_in, proj_dim)
            self.text_head  = LinearHead(text_in,  proj_dim)
        elif head_type == "mlp":
            self.audio_head = MLPHead(audio_in, proj_dim, CFG["mlp_hidden"], CFG["mlp_layers"], CFG["dropout"])
            self.text_head  = MLPHead(text_in,  proj_dim, CFG["mlp_hidden"], CFG["mlp_layers"], CFG["dropout"])
        elif head_type == "tx":
            self.audio_head = TxHead(audio_in, proj_dim, CFG["tx_tokens"], CFG["tx_layers"], CFG["tx_heads"], CFG["tx_ff"], CFG["dropout"])
            self.text_head  = TxHead(text_in,  proj_dim, CFG["tx_tokens"], CFG["tx_layers"], CFG["tx_heads"], CFG["tx_ff"], CFG["dropout"])
        else:
            raise ValueError("CFG['head_type'] invalid")

        if CFG["learnable_temperature"]:
            self.logit_scale = nn.Parameter(torch.tensor(math.log(1.0 / CFG["init_temperature"])))
        else:
            self.register_buffer("logit_scale", torch.tensor(math.log(1.0 / CFG["init_temperature"])))

    def forward(self, a, t):
        za = F.normalize(self.audio_head(a), dim=-1)
        zt = F.normalize(self.text_head(t),  dim=-1)
        s = self.logit_scale.exp().clamp(1e-3, 100.0)
        return za, zt, s


## Objective and metric

Train with symmetric audio→text and text→audio cross-entropy. Evaluate **audio→text only**, using the whole evaluation split as the candidate pool. `R@K` counts an exact paired-row match in the top K. Other captions for the same track are not accepted as positives by this metric.

In [ ]:
# contrastive objective and retrieval metrics
def contrastive_loss(za, zt, logit_scale, label_smoothing=0.0):
    logits = logit_scale * (za @ zt.t())
    n = logits.size(0)
    targets = torch.arange(n, device=logits.device)

    if label_smoothing > 0:
        ce_a = F.cross_entropy(logits, targets, label_smoothing=label_smoothing)
        ce_t = F.cross_entropy(logits.t(), targets, label_smoothing=label_smoothing)
    else:
        ce_a = F.cross_entropy(logits, targets)
        ce_t = F.cross_entropy(logits.t(), targets)

    return 0.5 * (ce_a + ce_t)

@torch.no_grad()
def recall_at_k(za, zt, ks=(1, 5, 10)):
    sim = za @ zt.t()
    gt = torch.arange(sim.size(0), device=sim.device)
    out = {}

    ranks = torch.argsort(sim, dim=1, descending=True)
    for k in ks:
        hit = (ranks[:, :k] == gt.unsqueeze(1)).any(dim=1).float().mean().item()
        out[f"R@{k}"] = hit

    return out


## Embedding datasets and loaders

In [ ]:
# dataloaders for cached embedding pairs (audio, text)
from torch.utils.data import Dataset, DataLoader

class EmbPairDS(Dataset):
    def __init__(self, A, T):
        self.A = A
        self.T = T

    def __len__(self):
        return self.A.shape[0]

    def __getitem__(self, i):
        return {"a": self.A[i].float(), "t": self.T[i].float()}

train_ds = EmbPairDS(mc_train_emb["audio"], mc_train_emb["text"])
val_ds = EmbPairDS(mc_val_emb["audio"], mc_val_emb["text"])

train_loader = DataLoader(
    train_ds,
    batch_size=CFG["batch_size"],
    shuffle=True,
    drop_last=True
)
val_loader = DataLoader(
    val_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    drop_last=False
)

audio_in = train_ds[0]["a"].numel()
text_in = train_ds[0]["t"].numel()
print("audio_in:", audio_in, "text_in:", text_in)


## Main training run

AdamW, gradient clipping and early stopping on validation R@10. The best checkpoint, last checkpoint, configuration and history are written to this run's directory. This run is separate from the three-model comparison below.

In [ ]:
from tqdm.auto import tqdm
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import torch

# JSON serialization helper for paths, numpy types, and tensors
def _json_safe(o):
    if isinstance(o, Path):
        return str(o)
    if isinstance(o, torch.Tensor):
        return o.detach().cpu().tolist()
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.ndarray,)):
        return o.tolist()
    return str(o)

# training control
force_full_epochs = False
metric_name = "R@10"

# dataloader setup
drop_last_train = (len(train_ds) >= CFG["batch_size"])
train_loader = DataLoader(
    train_ds,
    batch_size=CFG["batch_size"],
    shuffle=True,
    drop_last=drop_last_train
)
val_loader = DataLoader(
    val_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    drop_last=False
)

print("len(train_ds):", len(train_ds), "| len(val_ds):", len(val_ds))
print("batch_size:", CFG["batch_size"], "| drop_last_train:", drop_last_train)
print("len(train_loader):", len(train_loader), "| len(val_loader):", len(val_loader))
assert len(train_loader) > 0, "train_loader has 0 batches (check embeddings or batch_size)."

# model + optimizer
model = CLAPAdapter(audio_in=audio_in, text_in=text_in, proj_dim=CFG["proj_dim"]).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])

# mixed precision on CUDA only
use_amp = CFG["amp"] and (device.type == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda" and use_amp))

# tracking and checkpoints
best_val = -1.0
no_improve = 0
history = {
    "run_id": RUN_ID,
    "cfg": dict(CFG),
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_R@1": [],
    "val_R@5": [],
    "val_R@10": [],
}

best_path = RUN_DIR / "best_model.pt"
last_path = RUN_DIR / "last_checkpoint.pt"
hist_path = RUN_DIR / "history.json"

t0 = start_tracking()

try:
    epochs_ran = 0
    for epoch in tqdm(range(1, CFG["epochs"] + 1), desc="Training epochs"):
        epochs_ran += 1
        model.train()
        tr_losses = []

        for b in train_loader:
            a = b["a"].to(device)
            t = b["t"].to(device)

            opt.zero_grad(set_to_none=True)

            if device.type == "cuda":
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                    za, zt, s = model(a, t)
                    loss = contrastive_loss(za, zt, s, CFG["label_smoothing"])
                if not torch.isfinite(loss):
                    print("[WARN] Non-finite loss (CUDA). Skipping batch.")
                    continue

                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=float(CFG["grad_clip_norm"]))
                scaler.step(opt)
                scaler.update()
            else:
                za, zt, s = model(a, t)
                loss = contrastive_loss(za, zt, s, CFG["label_smoothing"])
                if not torch.isfinite(loss):
                    print("[WARN] Non-finite loss (MPS/CPU). Skipping batch.")
                    continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=float(CFG["grad_clip_norm"]))
                opt.step()

            tr_losses.append(loss.item())

        if len(tr_losses) == 0:
            print("[ERROR] No valid training batches (non-finite loss or empty loader).")
            break

        model.eval()
        va_losses = []
        all_za, all_zt = [], []

        for b in val_loader:
            a = b["a"].to(device)
            t = b["t"].to(device)
            with torch.no_grad():
                za, zt, s = model(a, t)
                loss = contrastive_loss(za, zt, s, CFG["label_smoothing"])
            va_losses.append(loss.item())
            all_za.append(za.detach())
            all_zt.append(zt.detach())

        all_za = torch.cat(all_za, dim=0)
        all_zt = torch.cat(all_zt, dim=0)
        rec = recall_at_k(all_za, all_zt, ks=(1, 5, 10))

        history["epoch"].append(epoch)
        history["train_loss"].append(float(np.mean(tr_losses)))
        history["val_loss"].append(float(np.mean(va_losses)))
        history["val_R@1"].append(rec["R@1"])
        history["val_R@5"].append(rec["R@5"])
        history["val_R@10"].append(rec["R@10"])

        metric = rec[metric_name]
        if metric > best_val:
            best_val = metric
            no_improve = 0
            torch.save(model.state_dict(), best_path)
        else:
            no_improve += 1

        torch.save(
            {"epoch": epoch, "model": model.state_dict(), "opt": opt.state_dict(), "history": history},
            last_path
        )
        hist_path.write_text(
            json.dumps(history, indent=2, ensure_ascii=False, default=_json_safe),
            encoding="utf-8"
        )

        print(
            f"epoch {epoch:03d} | train_loss={history['train_loss'][-1]:.4f} | "
            f"val_loss={history['val_loss'][-1]:.4f} | "
            f"R@1={rec['R@1']:.3f} R@5={rec['R@5']:.3f} R@10={rec['R@10']:.3f} | "
            f"best{metric_name}={best_val:.3f} | no_improve={no_improve}/{CFG['patience']}"
        )

        if (not force_full_epochs) and (no_improve >= CFG["patience"]):
            print("early stop: no improvement")
            break
finally:
    run_summary = stop_tracking(t0)
    (RUN_DIR / "training_resources.json").write_text(json.dumps(run_summary, indent=2))

print("epochs_ran:", epochs_ran)
run_summary


## Learning curves

In [ ]:
# learning curves
epochs = history["epoch"]

plt.figure(figsize=(8, 4))
plt.plot(epochs, history["train_loss"], label="train_loss")
plt.plot(epochs, history["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Contrastive loss")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(epochs, history["val_R@1"], label="val R@1")
plt.plot(epochs, history["val_R@5"], label="val R@5")
plt.plot(epochs, history["val_R@10"], label="val R@10")
plt.xlabel("Epoch")
plt.ylabel("Recall")
plt.title("Retrieval metrics (validation)")
plt.legend()
plt.tight_layout()
plt.show()


## Song Describer: main run

In [ ]:
# evaluation on Song Describer
model.load_state_dict(torch.load(best_path, map_location="cpu"))
model.to(device).eval()

A = sd_emb["audio"].float().to(device)
T = sd_emb["text"].float().to(device)

with torch.no_grad():
    za, zt, s = model(A, T)

sd_metrics = recall_at_k(za, zt, ks=(1, 5, 10))
sd_metrics


## Head comparison

Train a separate instance of each head on the same cached data. Select each checkpoint by validation R@10. The original comparison ran the heads sequentially without resetting the seed between them; this code retains that protocol. It is a single comparison without repeated seeds or uncertainty estimates.

In [ ]:
import copy

def train_one_head(head_type: str):
    # temporarily switch head type for comparison
    old = CFG["head_type"]
    CFG["head_type"] = head_type

    try:
        m = CLAPAdapter(audio_in=audio_in, text_in=text_in, proj_dim=CFG["proj_dim"]).to(device)
    finally:
        CFG["head_type"] = old
    opt = torch.optim.AdamW(m.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])

    use_amp_local = CFG["amp"] and (device.type == "cuda")
    scaler_local = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda" and use_amp_local))

    best = -1.0
    no_imp = 0
    best_state = None

    for epoch in range(1, CFG["epochs"] + 1):
        m.train()
        for b in train_loader:
            a = b["a"].to(device)
            t = b["t"].to(device)
            opt.zero_grad(set_to_none=True)

            if device.type == "cuda":
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp_local):
                    za, zt, s = m(a, t)
                    loss = contrastive_loss(za, zt, s, CFG["label_smoothing"])
                if not torch.isfinite(loss):
                    continue
                scaler_local.scale(loss).backward()
                scaler_local.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=float(CFG["grad_clip_norm"]))
                scaler_local.step(opt)
                scaler_local.update()
            else:
                za, zt, s = m(a, t)
                loss = contrastive_loss(za, zt, s, CFG["label_smoothing"])
                if not torch.isfinite(loss):
                    continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=float(CFG["grad_clip_norm"]))
                opt.step()

        # validation retrieval
        m.eval()
        all_za, all_zt = [], []
        with torch.no_grad():
            for b in val_loader:
                a = b["a"].to(device)
                t = b["t"].to(device)
                za, zt, s = m(a, t)
                all_za.append(za.detach())
                all_zt.append(zt.detach())

        rec = recall_at_k(torch.cat(all_za, 0), torch.cat(all_zt, 0), ks=(1, 5, 10))
        metric = rec["R@10"]

        if metric > best:
            best = metric
            no_imp = 0
            best_state = copy.deepcopy(m.state_dict())
        else:
            no_imp += 1

        if no_imp >= CFG["patience"]:
            break

    # evaluation on Song Describer
    if best_state is None:
        raise RuntimeError(f"No valid checkpoint for {head_type}.")
    m.load_state_dict(best_state)
    m.to(device).eval()

    A = sd_emb["audio"].float().to(device)
    T = sd_emb["text"].float().to(device)

    with torch.no_grad():
        za, zt, s = m(A, T)

    sd = recall_at_k(za, zt, ks=(1, 5, 10))

    CFG["head_type"] = old

    return {
        "head_type": head_type,
        "val_best_R@10": float(best),
        "songdesc_R@1": sd["R@1"],
        "songdesc_R@5": sd["R@5"],
        "songdesc_R@10": sd["R@10"],
    }

rows = [train_one_head(ht) for ht in ["linear", "mlp", "tx"]]
comp_df = pd.DataFrame(rows)
comp_df


## Text→audio demo

The free-text demo uses the reverse retrieval direction from the reported metric. It prints candidate row indices and captions; it does not embed audio in the saved notebook. Similarity scores are not probabilities.

In [ ]:
from IPython.display import Audio, display

@torch.no_grad()
def retrieve_from_query(query, topk=5):
    model.eval()
    # text query embedding and projection
    temb = embed_text_batch([query]).float().to(device)
    zq = F.normalize(model.text_head(temb), dim=-1)

    # project SongDescriber audio embeddings once per query
    A = sd_emb["audio"].float().to(device)
    za = F.normalize(model.audio_head(A), dim=-1)

    # similarity search in the shared embedding space
    sim = (zq @ za.t()).squeeze(0)
    idx = torch.argsort(sim, descending=True)[:topk].detach().cpu().numpy().tolist()
    return idx, sim.detach().cpu().numpy()

query = "uplifting danceable track with synths and a male singer"
top_idx, _ = retrieve_from_query(query, topk=5)

print("Query:", query)
print("Top idx:", top_idx)
kept_rows = source_row_indices(sd_emb["meta"], len(sd_eval))
for rank, embedding_index in enumerate(top_idx, start=1):
    row_index = kept_rows[embedding_index]
    print(rank, "source row", row_index, sd_eval.select_columns([sd_text_col])[row_index][sd_text_col])


## Save the run summary

In [ ]:
summary = {
    "config": CFG,
    "device": str(device),
    "torch_version": str(torch.__version__),
    "epochs_ran": len(history["epoch"]),
    "validation_best_R@10": max(history["val_R@10"]),
    "songdescriber_audio_to_text": sd_metrics,
    "evaluation": "exact paired row; full candidate pool; audio-to-text",
    "source_rows": {"musiccaps_train": len(mc_train), "musiccaps_validation": len(mc_val), "songdescriber": len(sd_eval)},
    "kept_rows": {"musiccaps_train": len(mc_train_emb["meta"]), "musiccaps_validation": len(mc_val_emb["meta"]), "songdescriber": len(sd_emb["meta"])},
    "dataset_fingerprints": {"train": mc_train._fingerprint, "validation": mc_val._fingerprint, "songdescriber": sd_eval._fingerprint},
    "embedding_resources": track_summary,
    "training_resources": run_summary,
}
(RUN_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
summary


## Interpretation

In the saved comparison, MLP Song Describer R@10 was 0.076408, linear was 0.073727 and transformer was 0.050938. The MLP advantage over linear was about 0.27 percentage points in that run. This is not evidence of a consistent advantage across seeds or datasets.

A separately trained main model reached 0.083110. Keep it separate from the comparison table. Short crops, the exact-row metric, repeated tracks and unrecorded historical model/data revisions limit interpretation. No generalization claim is made beyond these recorded experiments.